# Freight Forwarding Shipment Exception Monitoring

## Notebook 02 - Data Quality Assessment


## Objective

This notebook assesses the condition of the raw operational datasets before cleaning. The checks focus on dataset structure, completeness, duplicate records, categorical consistency, referential integrity, and selected business rules that can be evaluated by comparing existing raw values.

This notebook does not clean data, modify DataFrames, export files, perform SQL, create charts, or perform business analysis. Numeric and datetime-like columns are not parsed here; they are documented as raw source values and will be standardized during Notebook 03 (Data Cleaning).


## Import Libraries

Import pandas and the project raw data directory. The path setup allows the notebook to run from the `notebooks` folder while still importing the shared configuration module.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "scripts" / "config.py").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.config import RAW_DATA_DIR


## Load Datasets

Load each raw Excel file into a separate DataFrame. These DataFrames are treated as read-only inputs for the assessment.


In [2]:
shipments_df = pd.read_excel(RAW_DATA_DIR / "shipments.xlsx")
milestones_df = pd.read_excel(RAW_DATA_DIR / "shipment_milestones.xlsx")
issue_log_df = pd.read_excel(RAW_DATA_DIR / "issue_log.xlsx")


## Shipment Dataset Assessment

The Shipments dataset is the shipment-level reference table. It should contain one reliable record per shipment with the core fields needed for operational monitoring.


### Dataset Overview

Check the size and structure of the dataset. This matters because later checks depend on the expected columns being present and readable.


In [3]:
print(f"Rows: {shipments_df.shape[0]}")
print(f"Columns: {shipments_df.shape[1]}")

shipments_df.dtypes


Rows: 500
Columns: 26


Shipment ID              object
Booking Number           object
BL Number                object
Customer Name            object
Incoterm                 object
Cargo Type               object
Commodity                object
Cargo Volume CBM         object
Container Type           object
Container Qty             int64
Origin Port              object
Destination Port         object
Vessel Name              object
Shipping Line            object
ETD                      object
ETA                      object
Actual Departure         object
Actual Arrival           object
Cut Off Time             object
Closing Time             object
Total Charge USD         object
Shipment Status          object
Priority Level           object
PIC Name                 object
Created Date             object
Last Update Timestamp    object
dtype: object

### Completeness

Check whether required shipment fields contain missing values. The expected result is zero missing values in fields required for identification, routing, scheduling, and status monitoring.


In [4]:
required_columns = [
    "Shipment ID",
    "Booking Number",
    "Customer Name",
    "Origin Port",
    "Destination Port",
    "ETD",
    "ETA",
    "Shipment Status",
]

missing_counts = shipments_df[required_columns].isna().sum()
problem_count = int(missing_counts.sum())

print(f"Missing required values found: {problem_count}")

if problem_count > 0:
    display(missing_counts[missing_counts > 0])
    display(shipments_df[shipments_df[required_columns].isna().any(axis=1)].head())
else:
    print("Success: no missing values found in required shipment fields.")


Missing required values found: 0
Success: no missing values found in required shipment fields.


### Uniqueness

Check whether `Shipment ID` uniquely identifies shipment records and whether fully duplicated rows exist.


In [5]:
duplicate_shipments = shipments_df[
    shipments_df.duplicated(subset=["Shipment ID"], keep=False)
]
duplicate_rows = shipments_df[shipments_df.duplicated(keep=False)]

print(f"Duplicate Shipment ID records found: {duplicate_shipments.shape[0]}")
print(f"Fully duplicated rows found: {duplicate_rows.shape[0]}")

if not duplicate_shipments.empty:
    display(duplicate_shipments.head())
elif not duplicate_rows.empty:
    display(duplicate_rows.head())
else:
    print("Success: no duplicate shipment identifiers or rows found.")


Duplicate Shipment ID records found: 0
Fully duplicated rows found: 0
Success: no duplicate shipment identifiers or rows found.


### Consistency

Check distribution of important categorical fields using `value_counts()`. Unexpected categories, inconsistent labels, or unusually sparse categories should be reviewed before cleaning rules are approved.


In [6]:
shipment_categorical_fields = [
    "Shipment Status",
    "Priority Level",
    "Incoterm",
    "Cargo Type",
    "Container Type",
    "Origin Port",
    "Destination Port",
    "Shipping Line",
]

for column in shipment_categorical_fields:
    print(f"\n{column}")
    display(shipments_df[column].value_counts(dropna=False))



Shipment Status


Shipment Status
Customs Clearance    139
Delivered            124
In Transit           122
At Origin            115
Name: count, dtype: int64


Priority Level


Priority Level
Low        315
Medium     116
High        53
Low          4
 Low         3
low          2
Medium       1
 Medium      1
LOW          1
MEDIUM       1
medium       1
high         1
 High        1
Name: count, dtype: int64


Incoterm


...

Origin Port
Shanghai       93
Singapore      83
Hong Kong      77
Shenzhen       74
Busan          69
Port Klang     66
Hong Kong       3
SHANGHAI        3
Port Klangg     3
Singpore        2
 Port Klang     2
SHENZHEN        2
hong kong       2
Port  Klang     2
Busan           2
 Shanghai       2
BUSAN           2
Singapore       1
PORT KLANG      1
Shanghai        1
singapore       1
SINGAPORE       1
  Singapore     1
 Shenzhen       1
 Busan          1
shenzhen        1
 Singapore      1
Singapore       1
Hong  Kong      1
port klang      1
Name: count, dtype: int64


Destination Port


Destination Port
Jakarta     332
Surabaya     93
Semarang     75
Name: count, dtype: int64


Shipping Line


Shipping Line
OceanLink Shipping            110
BlueWave Lines                 97
Pacific Horizon Shipping       96
Global Route Shipping          27
Maritime Connect Lines         27
TransMarine Logistics          24
Eastern Cargo Line             24
Northstar Freight Lines        20
OceanVista Carriers            20
Archipelago Maritime           14
bluewave lines                  5
Pacific  Horizon  Shipping      4
OceanLink  Shipping             4
pacific horizon shipping        3
 OceanLink Shipping             3
oceanlink shipping              2
BlueWave Lines                  2
BlueWave  Lines                 2
OCEANLINK SHIPPING              1
 Global Route Shipping          1
 Pacific Horizon Shipping       1
BLUEWAVE LINES                  1
OceanLink Shipping              1
PACIFIC HORIZON SHIPPING        1
Eastern  Cargo  Line            1
 BlueWave Lines                 1
GLOBAL ROUTE SHIPPING           1
EASTERN CARGO LINE              1
Pacific Horizon Shipping        1


### Validity

Date-like and numeric-like shipment fields are assessed only as raw source values in this notebook. No parsing or type standardization is performed here. These fields are currently stored as raw text in the source extract and will be standardized during Notebook 03 (Data Cleaning).


In [7]:
raw_text_fields = [
    "ETD",
    "ETA",
    "Actual Departure",
    "Actual Arrival",
    "Cut Off Time",
    "Closing Time",
    "Created Date",
    "Last Update Timestamp",
    "Cargo Volume CBM",
    "Container Qty",
    "Total Charge USD",
]

shipments_df[raw_text_fields].dtypes


ETD                      object
ETA                      object
Actual Departure         object
Actual Arrival           object
Cut Off Time             object
Closing Time             object
Created Date             object
Last Update Timestamp    object
Cargo Volume CBM         object
Container Qty             int64
Total Charge USD         object
dtype: object

### Business Rule Validation

Check selected shipment lifecycle rules that can be evaluated by comparing existing raw values. The expected result is zero delivered shipments without `Actual Arrival`.


In [8]:
delivered_without_arrival = shipments_df[
    shipments_df["Shipment Status"].str.lower().eq("delivered")
    & shipments_df["Actual Arrival"].isna()
]

print(
    "Delivered shipments without Actual Arrival found: "
    f"{delivered_without_arrival.shape[0]}"
)

if not delivered_without_arrival.empty:
    display(delivered_without_arrival.head())
else:
    print("Success: all delivered shipments have Actual Arrival populated.")


Delivered shipments without Actual Arrival found: 0
Success: all delivered shipments have Actual Arrival populated.


### Shipment Findings

Review the shipment assessment output above and summarize the findings.

- Completeness:
- Uniqueness:
- Consistency:
- Validity:
- Recommended Cleaning Actions:


## Shipment Milestones Assessment

The Shipment Milestones dataset records operational events for each shipment. It should support event tracking, sequence checks, and linkage back to the shipment master data.


### Dataset Overview

Check the size and structure of the dataset. This matters because milestone monitoring depends on expected event columns and shipment references.


In [9]:
print(f"Rows: {milestones_df.shape[0]}")
print(f"Columns: {milestones_df.shape[1]}")

milestones_df.dtypes


Rows: 5000
Columns: 10


Milestone ID          object
Shipment ID           object
Milestone Sequence     int64
Milestone Type        object
Planned Datetime      object
Actual Datetime       object
Milestone Status      object
Updated By            object
Update Notes          object
Update Timestamp      object
dtype: object

### Completeness

Check whether required milestone fields contain missing values. The expected result is zero missing values in fields required for event identification, sequence tracking, and shipment linkage.


In [10]:
required_columns = [
    "Milestone ID",
    "Shipment ID",
    "Milestone Sequence",
    "Milestone Type",
    "Planned Datetime",
    "Actual Datetime",
    "Milestone Status",
]

missing_counts = milestones_df[required_columns].isna().sum()
problem_count = int(missing_counts.sum())

print(f"Missing required values found: {problem_count}")

if problem_count > 0:
    display(missing_counts[missing_counts > 0])
    display(milestones_df[milestones_df[required_columns].isna().any(axis=1)].head())
else:
    print("Success: no missing values found in required milestone fields.")


Missing required values found: 0
Success: no missing values found in required milestone fields.


### Uniqueness

Check whether `Milestone ID` uniquely identifies milestone records and whether fully duplicated rows exist.


In [11]:
duplicate_milestones = milestones_df[
    milestones_df.duplicated(subset=["Milestone ID"], keep=False)
]
duplicate_rows = milestones_df[milestones_df.duplicated(keep=False)]

print(f"Duplicate Milestone ID records found: {duplicate_milestones.shape[0]}")
print(f"Fully duplicated rows found: {duplicate_rows.shape[0]}")

if not duplicate_milestones.empty:
    display(duplicate_milestones.head())
elif not duplicate_rows.empty:
    display(duplicate_rows.head())
else:
    print("Success: no duplicate milestone identifiers or rows found.")


Duplicate Milestone ID records found: 0
Fully duplicated rows found: 0
Success: no duplicate milestone identifiers or rows found.


### Consistency

Check milestone sequence repetition and category distributions using `value_counts()`. Unexpected milestone types, statuses, or repeated sequence values should be reviewed before cleaning rules are approved.


In [12]:
duplicate_sequences = milestones_df[
    milestones_df.duplicated(
        subset=["Shipment ID", "Milestone Sequence"],
        keep=False,
    )
]

print(
    "Duplicate milestone sequence records within shipment found: "
    f"{duplicate_sequences.shape[0]}"
)

if not duplicate_sequences.empty:
    display(duplicate_sequences.head())
else:
    print("Success: no duplicate milestone sequences within shipment found.")


Duplicate milestone sequence records within shipment found: 0
Success: no duplicate milestone sequences within shipment found.


In [13]:
milestone_categorical_fields = [
    "Milestone Type",
    "Milestone Status",
    "Updated By",
]

for column in milestone_categorical_fields:
    print(f"\n{column}")
    display(milestones_df[column].value_counts(dropna=False))



Milestone Type


Milestone Type
Cargo Ready               500
Booking Confirmed         500
Empty Container Pickup    500
Cargo Loading             500
Gate In                   500
Documentation Complete    500
Vessel Departure          500
Port Arrival              500
Customs Clearance         500
Delivered                 500
Name: count, dtype: int64


Milestone Status


Milestone Status
Completed    2426
Delayed      2231
completed     114
delayed        86
DELAYED        48
COMPLETED      45
pending        26
PENDING        24
Name: count, dtype: int64


Updated By


Updated By
Andi Pratama       636
Budi Santoso       593
Cindy Wijaya       534
Dimas Saputra      493
Fajar Nugroho      465
Indah Permata      392
Kevin Halim        367
Maya Putri         341
Rizky Kurniawan    280
Sarah Amelia       219
Daniel Hartono     152
NaN                130
Nadia Prasetyo     115
Rafi Kurnia        101
Michael Gunawan     93
Jessica Putri       89
Name: count, dtype: int64

### Validity

Date-like and numeric-like milestone fields are assessed only as raw source values in this notebook. No parsing or type standardization is performed here. These fields are currently stored as raw text in the source extract and will be standardized during Notebook 03 (Data Cleaning).


In [14]:
raw_text_fields = [
    "Milestone Sequence",
    "Planned Datetime",
    "Actual Datetime",
    "Update Timestamp",
]

milestones_df[raw_text_fields].dtypes


Milestone Sequence     int64
Planned Datetime      object
Actual Datetime       object
Update Timestamp      object
dtype: object

### Referential Integrity

Check whether every milestone references a shipment that exists in the Shipments dataset. The expected result is zero milestone records with unknown `Shipment ID` values.


In [15]:
milestones_with_unknown_shipments = milestones_df[
    ~milestones_df["Shipment ID"].isin(shipments_df["Shipment ID"])
]

print(
    "Milestone records with unknown Shipment ID found: "
    f"{milestones_with_unknown_shipments.shape[0]}"
)

if not milestones_with_unknown_shipments.empty:
    display(milestones_with_unknown_shipments.head())
else:
    print("Success: all milestone Shipment ID values exist in Shipments.")


Milestone records with unknown Shipment ID found: 0
Success: all milestone Shipment ID values exist in Shipments.


### Business Rule Validation

Check selected milestone lifecycle rules that can be evaluated by comparing existing raw values. The expected result is zero completed milestones without `Actual Datetime`.


In [16]:
completed_without_actual_datetime = milestones_df[
    milestones_df["Milestone Status"].str.lower().eq("completed")
    & milestones_df["Actual Datetime"].isna()
]

print(
    "Completed milestones without Actual Datetime found: "
    f"{completed_without_actual_datetime.shape[0]}"
)

if not completed_without_actual_datetime.empty:
    display(completed_without_actual_datetime.head())
else:
    print("Success: all completed milestones have Actual Datetime populated.")


Completed milestones without Actual Datetime found: 0
Success: all completed milestones have Actual Datetime populated.


### Milestone Findings

Review the milestone assessment output above and summarize the findings.

- Completeness:
- Uniqueness:
- Consistency:
- Validity:
- Recommended Cleaning Actions:


## Issue Log Assessment

The Issue Log dataset records operational exceptions and resolution activity. It should support issue tracking, lifecycle checks, and linkage back to shipment records.


### Dataset Overview

Check the size and structure of the dataset. This matters because exception monitoring depends on expected issue, status, date, and shipment reference fields.


In [17]:
print(f"Rows: {issue_log_df.shape[0]}")
print(f"Columns: {issue_log_df.shape[1]}")

issue_log_df.dtypes


Rows: 750
Columns: 14


Issue ID                 object
Shipment ID              object
Issue Type               object
Issue Description        object
Severity Level           object
Issue Status             object
Issue Open Date          object
Due Date                 object
Issue Closed Date        object
Assigned PIC             object
Corrective Action        object
Root Cause Category      object
Root Cause Notes         object
Last Update Timestamp    object
dtype: object

### Completeness

Check whether required issue fields contain missing values. The expected result is zero missing values in fields required for issue identification, status monitoring, and shipment linkage.


In [18]:
required_columns = [
    "Issue ID",
    "Shipment ID",
    "Issue Type",
    "Issue Description",
    "Severity Level",
    "Issue Status",
    "Issue Open Date",
    "Assigned PIC",
]

missing_counts = issue_log_df[required_columns].isna().sum()
problem_count = int(missing_counts.sum())

print(f"Missing required values found: {problem_count}")

if problem_count > 0:
    display(missing_counts[missing_counts > 0])
    display(issue_log_df[issue_log_df[required_columns].isna().any(axis=1)].head())
else:
    print("Success: no missing values found in required issue fields.")


Missing required values found: 0
Success: no missing values found in required issue fields.


### Uniqueness

Check whether `Issue ID` uniquely identifies issue records and whether fully duplicated rows exist.


In [19]:
duplicate_issues = issue_log_df[
    issue_log_df.duplicated(subset=["Issue ID"], keep=False)
]
duplicate_rows = issue_log_df[issue_log_df.duplicated(keep=False)]

print(f"Duplicate Issue ID records found: {duplicate_issues.shape[0]}")
print(f"Fully duplicated rows found: {duplicate_rows.shape[0]}")

if not duplicate_issues.empty:
    display(duplicate_issues.head())
elif not duplicate_rows.empty:
    display(duplicate_rows.head())
else:
    print("Success: no duplicate issue identifiers or rows found.")


Duplicate Issue ID records found: 0
Fully duplicated rows found: 0
Success: no duplicate issue identifiers or rows found.


### Consistency

Check distribution of important categorical fields using `value_counts()`. Unexpected issue types, statuses, severity levels, or root-cause categories should be reviewed before cleaning rules are approved.


In [20]:
issue_categorical_fields = [
    "Issue Type",
    "Severity Level",
    "Issue Status",
    "Assigned PIC",
    "Root Cause Category",
]

for column in issue_categorical_fields:
    print(f"\n{column}")
    display(issue_log_df[column].value_counts(dropna=False))



Issue Type


Issue Type
Documentation        162
Port Congestion      155
Weather Delay        153
Amendment Request    147
Customs Hold         133
Name: count, dtype: int64


Severity Level


Severity Level
Critical    205
Low         192
High        185
Medium      168
Name: count, dtype: int64


Issue Status


Issue Status
Open      395
Closed    355
Name: count, dtype: int64


Assigned PIC


Assigned PIC
Budi Santoso       104
Andi Pratama        99
Fajar Nugroho       84
Cindy Wijaya        81
Dimas Saputra       78
Maya Putri          57
Kevin Halim         50
Indah Permata       50
Rizky Kurniawan     38
Sarah Amelia        36
Michael Gunawan     24
Daniel Hartono      14
Rafi Kurnia         14
Nadia Prasetyo      12
Jessica Putri        9
Name: count, dtype: int64


Root Cause Category


Root Cause Category
Equipment        109
Customs          100
Terminal          98
Port              96
Carrier           82
Customer          79
Documentation     69
Weather           59
Process           58
Name: count, dtype: int64

### Validity

Date-like issue fields are assessed only as raw source values in this notebook. No parsing or type standardization is performed here. These fields are currently stored as raw text in the source extract and will be standardized during Notebook 03 (Data Cleaning).


In [21]:
raw_text_fields = [
    "Issue Open Date",
    "Due Date",
    "Issue Closed Date",
    "Last Update Timestamp",
]

issue_log_df[raw_text_fields].dtypes


Issue Open Date          object
Due Date                 object
Issue Closed Date        object
Last Update Timestamp    object
dtype: object

### Referential Integrity

Check whether every issue references a shipment that exists in the Shipments dataset. The expected result is zero issue records with unknown `Shipment ID` values.


In [22]:
issues_with_unknown_shipments = issue_log_df[
    ~issue_log_df["Shipment ID"].isin(shipments_df["Shipment ID"])
]

print(
    "Issue records with unknown Shipment ID found: "
    f"{issues_with_unknown_shipments.shape[0]}"
)

if not issues_with_unknown_shipments.empty:
    display(issues_with_unknown_shipments.head())
else:
    print("Success: all issue Shipment ID values exist in Shipments.")


Issue records with unknown Shipment ID found: 0
Success: all issue Shipment ID values exist in Shipments.


### Business Rule Validation

Check selected issue lifecycle rules that can be evaluated by comparing existing raw values. The expected result is zero closed issues without `Issue Closed Date`.


In [23]:
closed_without_closed_date = issue_log_df[
    issue_log_df["Issue Status"].str.lower().eq("closed")
    & issue_log_df["Issue Closed Date"].isna()
]

print(
    "Closed issues without Issue Closed Date found: "
    f"{closed_without_closed_date.shape[0]}"
)

if not closed_without_closed_date.empty:
    display(closed_without_closed_date.head())
else:
    print("Success: all closed issues have Issue Closed Date populated.")


Closed issues without Issue Closed Date found: 0
Success: all closed issues have Issue Closed Date populated.


### Issue Log Findings

Review the issue log assessment output above and summarize the findings.

- Completeness:
- Uniqueness:
- Consistency:
- Validity:
- Recommended Cleaning Actions:


## Overall Assessment

Review the outputs from all three dataset sections and summarize the overall data quality position.

- Completeness:
- Uniqueness:
- Consistency:
- Validity:
- Recommended Cleaning Actions:


## Next Step

Review the assessment outputs with business and source-system owners. Once findings are confirmed, define the approved cleaning rules for Notebook 03 (Data Cleaning).
